Sentiment Analysis Using conditional Workflow

In [63]:
from langgraph.graph import StateGraph, START, END
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from typing import TypedDict, Annotated, Literal
import operator
import os

In [64]:
load_dotenv()

llm = ChatOpenAI()

In [65]:
class SentimentSchema(BaseModel):
    sentiment: Literal['Positive','Negative'] = Field(description='sentiment of user review')

llm_with_structured_output = llm.with_structured_output(SentimentSchema)

class DiagnosisSchema(BaseModel):
    issue_type: Literal["UX", "Performance", "Bug", "Support", "Other"] = Field(description='The category of issue mentioned in the review')
    tone: Literal["angry", "frustrated", "disappointed", "calm"] = Field(description='The emotional tone expressed by the user')
    urgency: Literal["low", "medium", "high"] = Field(description='How urgent or critical the issue appears to be')


llm_with_structured_output2 = llm.with_structured_output(DiagnosisSchema)

In [66]:
#define state
class FeedbackState(TypedDict):
    review:str
    sentiment: Literal['Positive','Negative']
    diagnosis: dict
    response: str



In [67]:
def find_sentiment(state:FeedbackState ):
    """This function analyses a review and find sentiment , provides answer in positive or negative"""

    prompt = f"""
        Analyse the review {state['review']} and provide the sentiment either 'positive' or 'negative'
    """
    result = llm_with_structured_output.invoke(prompt)
    return {'sentiment' : result.sentiment}

def diagnosis_review(state:FeedbackState) : 
    """This function analyses, diagnoses the review and find the issue type, tone, urgency """

    prompt = f"""
        Generate a diagnosis report of the review {state['review']} and find the issue type, tone , urgency
    """
    result = llm_with_structured_output2.invoke(prompt)
    return {'diagnosis' : result.model_dump()}


def positive_response(state:FeedbackState) : 
    """This function writes a reply when user review is positive"""

    prompt = f"""
        Generate a response based on the positive sentiment :  {state['sentiment']}'
    """
    result = llm.invoke(prompt)
    return {'response' : result.content}


def negative_response(state:FeedbackState) : 
    """This function writes a reply when user review is negative"""

    diagnosis = state['diagnosis']
    prompt = f"""
        Generate a reply based on the sentiment diagnosis - issue type : {diagnosis['issue_type']}, tone : {diagnosis['tone']}, urgency : {diagnosis['urgency']}'
    """
    result = llm.invoke(prompt)
    return {'response' : result.content}

def check_sentiment(state:FeedbackState) -> Literal['positive_response','diagnosis_review'] : 
    """This function checks the sentiment whether it is positive or negative"""
    if state['sentiment'] == 'Positive':
        return 'positive_response'
    else:
        return 'diagnosis_review'
    



In [68]:
graph = StateGraph(FeedbackState)

graph.add_node('find_sentiment',find_sentiment)
graph.add_node('positive_response', positive_response)
graph.add_node('negative_response', negative_response)
graph.add_node('diagnosis_review', diagnosis_review)

graph.add_edge(START, 'find_sentiment')
graph.add_conditional_edges('find_sentiment', check_sentiment)
graph.add_edge('diagnosis_review', 'negative_response')
graph.add_edge('negative_response',END)
graph.add_edge('positive_response',END)

workflow = graph.compile()

In [69]:
#initial_state={'review' :  "I’ve been trying to log in for over an hour now, and the app keeps freezing on the authentication screen. I even tried reinstalling it, but no luck. This kind of bug is unacceptable, especially when it affects basic functionality."}
initial_state={'review':"This is a very good product"}
output_state=workflow.invoke(initial_state)
print(output_state)

{'review': 'This is a very good product', 'sentiment': 'Positive', 'response': "That's great to hear! It's always refreshing to encounter positivity and optimism. Keep spreading those good vibes and embracing the positive energy around you. Keep shining bright!"}
